# 10. Lite Distill: grid-head, DX policy и π-Flow

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 1](https://diffusion.csail.mit.edu/2026/docs/20260120_Lecture_01.pdf) · [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf).
<!-- study-navigation:end -->

**Результат:** разобрать 160/400 outputs, выполнить native grid forward и отдельно network-free policy rollout

**Перед началом:** K6 08–09; ODE и Euler. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/07_sampling.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §4.5, 5](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/piflow_dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/piflow_dit.py)
- [`core/algo/piflow_math.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/piflow_math.py)
- [`core/algo/piflow_sampler.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/piflow_sampler.py)
- [`runtime/sampler.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/runtime/sampler.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','main'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '10_piflow_distillation'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Distillation меняет контракт головы

Обычный FM backbone возвращает один velocity per modality. PiFlow wrapper предсказывает
G grid outputs: video G×16, audio G×40. При G=10 это 160/400, но физическое состояние остаётся 16/40.
Wrapper делает reshape и movedim: `[B,G,...,C]`.

**Два способа конфигурации нельзя смешивать:** HF config уже хранит расширенные 160/400;
native PiFlow constructor принимает per-grid 16/40 и сам умножает на G. Наш tiny constructor
использует per-grid 4/4 и G=3; это другой размер, но тот же native wrapper.

Первичный источник: [π-Flow / Policy-Based Few-Step Generation](https://arxiv.org/abs/2510.14974).
Distillation обучает student воспроизводить поведение teacher. Простое уменьшение числа Euler steps
без обучения student не является distillation. История K6 §4.5 включает π-Flow и adversarial post-training;
здесь исследуем released inference math, не воспроизводим industrial distillation training.


In [ ]:
model=tiny_model(grid=3).eval();v,a,kw=tiny_inputs(model)
with torch.no_grad():pv,pa=fresh_forward(model,kw)
assert pv.shape==(1,3,2,4,4,4) and pa.shape==(1,3,6,4)
metrics['native_grid_shapes']=[list(pv.shape),list(pa.shape)]
print('Physical state:',v.shape,a.shape,'Grid predictions:',pv.shape,pa.shape)
show_source('core/components/piflow_dit.py','PiFlowDiffusionTransformer3D')


## 2. Из grid prediction в DX policy

Network вычисляется один раз на начале сегмента. Для каждого grid output строится candidate clean endpoint:

$$
\widehat x_0^{(g)}=x_{\sigma_s}-\sigma_s\widehat u^{(g)}.
$$

`DXPolicy` интерполирует эти endpoints по **raw time** внутри сегмента и возвращает velocity:

$$
\pi(x,\sigma)=\frac{x-\widehat x_0(\tau)}{\max(\sigma,\varepsilon)},\qquad
\sigma=s(\tau)=\frac{a\tau}{1+(a-1)\tau}.
$$

Raw tau и shifted sigma — разные координаты времени. `policy_rollout_fm` делает много дешёвых
подшагов этой функции **без нового transformer forward на каждом подшаге**.
Число model NFE, grid points и policy substeps — три разных числа.


In [ ]:
mathmod=native('core.algo.piflow_math')
x0=torch.tensor([[.3,-.2]],dtype=torch.float64);noise=torch.tensor([[1.2,.8]],dtype=torch.float64)
u=noise-x0;grid=u[:,None,:].expand(1,10,2).clone()
policy=mathmod.DXPolicy(grid,noise,torch.ones(1,dtype=torch.float64),segment_size=1.,shift=5.)
end,_,_=mathmod.policy_rollout_fm(noise,torch.ones(1,dtype=torch.float64),
     torch.ones(1,dtype=torch.float64),torch.zeros(1,dtype=torch.float64),128,policy)
error=(end-x0).abs().max().item();assert error<1e-10
metrics['oracle_constant_grid_endpoint_error']=error
raw=torch.linspace(0,1,101);shifted=mathmod.shift_timesteps(raw,5.)
fig,ax=plt.subplots(figsize=(6,3));ax.plot(raw,raw,label='raw');ax.plot(raw,shifted,label='shifted sigma')
ax.legend();ax.set_xlabel('Raw tau');fig.tight_layout();fig.savefig(figure_path(LAB,'policy_time'));plt.show()
print('Oracle grid control endpoint error:',error)


## 3. Native sampler + настоящий случайный tiny grid-DiT

Теперь исполним полный released **tensor sampler**, но на уменьшенной сети и случайных весах.
Это проверяет стыковку grid layout, двух policies и state shapes; осмысленное видео/аудио не синтезируется.
Stock loop проверяет, что присутствуют оба latent tensors. Отключить обновление video не равнозначно
удалить video input и fusion. Runtime запрещает PiFlow + CFG != 1 и PiFlow + MagCache/NaviCache.


In [ ]:
Bundle=native('core.types').LatentBundle
bundle=Bundle(v.clone(),a.clone(),torch.tensor([0,2],dtype=torch.int32),torch.tensor([0,6],dtype=torch.int32))
count=[];hook=model.register_forward_hook(lambda *args:count.append(1))
model.clear_text_proj_cache()
result=native('core.algo.piflow_sampler').piflow_denoise_loop(
    bundle,model,{'text_embeds':kw['text_embed'],'pooled_embed':kw['pooled_text_embed']},
    kw['visual_rope'],kw['audio_rope'],kw['text_rope'],num_steps=2,scheduler_scale=5.,
    attention_mask=kw['attention_mask'],num_policy_substeps=16)
hook.remove()
assert result.video.shape==v.shape and result.audio.shape==a.shape
assert torch.isfinite(result.video).all() and torch.isfinite(result.audio).all() and len(count)==2
metrics['native_tiny_sampler_model_NFE']=len(count)
metrics['native_tiny_sampler_finite']=True
print('2 transformer calls; policy substeps are not extra transformer calls')


## Самостоятельное исследование

1. В oracle опыте сделайте grid points различными и нарисуйте интерполируемый x0(tau).
2. Сравните 1, 16, 128 policy substeps при фиксированных network predictions; запишите NFE отдельно.
3. Объясните, почему переключить обычный Lite checkpoint на PiFlow scheduler недостаточно.
4. Найдите в sampler final_step_size_scale=.5: почему последний сегмент короче остальных?

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы отличаете физические latent channels, expanded output-head width, grid axis, network NFE
и network-free substeps. Для distill checkpoint используете соответствующий sampler contract.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 09. Joint Flow Matching: path, velocity и учебное обучение](09_flow_matching_training.ipynb) · [11. Sampling: Euler, CFG, schedules и NFE →](11_sampling_cfg_and_nfe.ipynb)
